# 20b — Pilota del few-shot: confronto fra embedding

Il notebook [20](20_fewshot_pilota.ipynb) ha scelto k e variante di prompt tenendo fisso
l'embedding, `all-mpnet-base-v2`, perché era il migliore nel pilota di Federica
(`notebooks/llm/20_preliminary_fs_test.ipynb`). Quel pilota però aveva due problemi che ne
invalidano il confronto fra embedding:

1. **Contesto di ollama a 4k.** I prompt few-shot (mediana ~12k token a k=4) venivano troncati
   in silenzio, quindi gran parte degli esempi non arrivava al modello.
2. **Righe del train fra quelle da riassumere.** 81 delle 100 righe di `sample_100_seed42.tsv`
   sono del train, cioè dello stesso archivio da cui si prendono gli esempi: per quelle righe il
   vicino più simile è la riga stessa, e il suo riassunto di riferimento finisce nel prompt.
   Tutti gli embedding la ritrovano, quindi il confronto misurava soprattutto quanto il modello
   ricopiava.

Qui il confronto si rifà con lo stesso protocollo del notebook 20: le stesse 100 righe di
**validation**, k=4, prompt `new`, `num_ctx=32768`, esempi troncati a 1.000 parole. Cambia
**solo** l'embedding con cui si cercano i vicini:

| embedding | modello | testo in ingresso |
|---|---|---|
| `sbert_mpnet_en` | `all-mpnet-base-v2` (SBERT inglese) | primi 384 token del cluster (limite del modello) |
| `sbert_multilingual` | `paraphrase-multilingual-mpnet-base-v2` | primi 128 token (limite del modello) |
| `tfidf` | `TfidfVectorizer(max_features=20000)` addestrato sul train | documento intero |
| `ollama_nomic` | `nomic-embed-text` via ollama | prime 512 parole, senza i prefissi di task |

Sono gli stessi quattro embedding e le stesse impostazioni del pilota di Federica, compresa
l'assenza dei prefissi `search_query:`/`search_document:` che `nomic-embed-text`
raccomanderebbe. `sbert_mpnet_en` non si rigenera: vicini e riassunti sono quelli del
notebook 20 (configurazione `k4_new`), identici per costruzione.

**Regola di decisione, fissata prima di eseguire.** Cambiare embedding vorrebbe dire rigenerare
le corse `test` e `test_budgetref` del notebook 21 (~35 h). Si cambia solo se un embedding
batte `sbert_mpnet_en` con una differenza **appaiata** di ROUGE-1 F1 il cui IC 95% esclude lo
zero. Altrimenti si tiene `all-mpnet-base-v2`, e le differenze si riportano come misurate.

In [ ]:
# --- Configurazione ---------------------------------------------------------
import hashlib
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

import summ_utils as su

N_PILOTA    = 100
SEED        = 42
MODELLO     = 'qwen2.5:7b-instruct'
MAX_TOKENS  = 200
TEMPERATURE = 0.3
K           = 4
VARIANTE    = 'new'
EMBEDDING   = ['sbert_mpnet_en', 'sbert_multilingual', 'tfidf', 'ollama_nomic']
RIFERIMENTO = 'sbert_mpnet_en'   # l'embedding in uso nel notebook 21
# SBERT multilingue: nome su Hugging Face, oppure una cartella locale se e' stato scaricato a
# mano (la CDN di Hugging Face e' bloccata su alcune reti): SBERT_MULTILINGUE=C:/modelli/...
MODELLO_MULTILINGUE = os.environ.get('SBERT_MULTILINGUE', 'paraphrase-multilingual-mpnet-base-v2')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
PILOTA_DIR = BASE / 'results' / 'fewshot' / f'pilota_val{N_PILOTA}'   # quella del notebook 20
EMB_DIR    = PILOTA_DIR / 'embedding'
CACHE_EMB  = BASE / 'results' / 'embeddings_cache'
RISULTATI_PATH = EMB_DIR / 'riepilogo.json'

# Stesse 100 righe del notebook 20 (stessa lista, stesso seme)
righe_val = list(su.itera_split(P['complete_tab'], 'val'))
righe = sorted(random.Random(SEED).sample(righe_val, N_PILOTA), key=lambda r: r['row_id'])
del righe_val
print(f'{len(righe)} righe di validation, row_id {righe[0]["row_id"]}..{righe[-1]["row_id"]}')
assert (righe[0]['row_id'], righe[-1]['row_id']) == (44933, 50488), 'righe diverse dal notebook 20'

def percorso_vicini(emb):
    return PILOTA_DIR / 'vicini.tsv' if emb == RIFERIMENTO else EMB_DIR / emb / 'vicini.tsv'

def percorso_riassunti(emb):
    return PILOTA_DIR / 'k4_new.tsv' if emb == RIFERIMENTO else EMB_DIR / emb / 'k4_new.tsv'

## Retrieval con ciascun embedding

Per ogni embedding: vettori del train (44.880 cluster) e delle 100 righe, 4 vicini per
coseno, TSV dei vicini in `results/fewshot/pilota_val100/embedding/{embedding}/vicini.tsv`.
I vettori densi restano in `results/embeddings_cache/` (non versionata); TF-IDF si ricalcola in
pochi minuti leggendo `complete.tab` in streaming, senza tenere in memoria i testi.

In [ ]:
def testi_train(completo=False):
    """Generatore dei testi del train, nell'ordine di complete.tab."""
    for es in su.itera_split(P['complete_tab'], 'train'):
        yield su.prepara_documento(es['document']) if completo else su.testo_per_embedding(es['document'])

train_ids = [es['row_id'] for es in su.itera_split(P['complete_tab'], 'train')]
query_ids = [r['row_id'] for r in righe]


def embedding_ollama(testi, etichetta, modello='nomic-embed-text', batch=64):
    """Embedding via /api/embed di ollama, normalizzati, con cache .npy come su.embedding_fewshot."""
    import requests
    h = hashlib.sha256(modello.encode('utf-8'))
    for t in testi:
        h.update(b'\x00')
        h.update(t.encode('utf-8'))
    path = CACHE_EMB / f'{etichetta}_{modello}_{h.hexdigest()[:16]}.npy'
    if path.exists():
        print(f'[{etichetta}] embedding dalla cache ({path.name})')
        return np.load(path)
    print(f'[{etichetta}] calcolo embedding di {len(testi)} testi con {modello}...')
    vettori = []
    for i in range(0, len(testi), batch):
        r = requests.post(f'{su.URL_OLLAMA_NATIVO}/api/embed', timeout=600,
                          json={'model': modello, 'input': testi[i:i + batch]})
        r.raise_for_status()
        vettori.extend(r.json()['embeddings'])
    emb = np.asarray(vettori, dtype=np.float32)
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    CACHE_EMB.mkdir(parents=True, exist_ok=True)
    np.save(path, emb)
    return emb


def vettori(emb):
    """(train, query) per l'embedding `emb`."""
    if emb == 'sbert_multilingual':
        modello = MODELLO_MULTILINGUE
        return (su.embedding_fewshot(list(testi_train()), CACHE_EMB, 'train', modello),
                su.embedding_fewshot([su.testo_per_embedding(r['document']) for r in righe],
                                     CACHE_EMB, f'pilota_val{N_PILOTA}', modello))
    if emb == 'ollama_nomic':
        return (embedding_ollama(list(testi_train()), 'train'),
                embedding_ollama([su.testo_per_embedding(r['document']) for r in righe],
                                 f'pilota_val{N_PILOTA}'))
    if emb == 'tfidf':
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(max_features=20000)
        print('[tfidf] addestramento sul train (documenti interi, due passate in streaming)...')
        vec.fit(testi_train(completo=True))
        return (vec.transform(testi_train(completo=True)),
                vec.transform([su.prepara_documento(r['document']) for r in righe]))
    raise ValueError(emb)


VICINI = {}
for emb in EMBEDDING:
    path = percorso_vicini(emb)
    if not path.exists():
        emb_train, emb_query = vettori(emb)
        indici, similarita = su.vicini_fewshot(emb_train, emb_query, K)
        su.salva_vicini(path, query_ids, train_ids, indici, similarita)
    VICINI[emb] = su.carica_vicini(path, k=K)
    print(f'{emb:20s} vicini: {path.relative_to(BASE)}')

ESEMPI_TRAIN = su.carica_esempi_train(P['complete_tab'],
                                      {v for vic in VICINI.values() for r in righe
                                       for v, _ in vic[r['row_id']]})
print(f'{len(ESEMPI_TRAIN)} esempi del train caricati')

### Quanto si somigliano i vicini

Prima di generare: quanti dei 4 vicini trovati da ciascun embedding coincidono con quelli di
`sbert_mpnet_en`. Se coincidono quasi tutti, le differenze di metrica fra embedding non possono
che essere piccole.

In [ ]:
for emb in EMBEDDING:
    if emb == RIFERIMENTO:
        continue
    comuni = [len({v for v, _ in VICINI[emb][r['row_id']]} & {v for v, _ in VICINI[RIFERIMENTO][r['row_id']]})
              for r in righe]
    primo = np.mean([VICINI[emb][r['row_id']][0][0] == VICINI[RIFERIMENTO][r['row_id']][0][0] for r in righe])
    print(f'{emb:20s} vicini in comune con {RIFERIMENTO}: media {np.mean(comuni):.2f}/{K}, '
          f'stesso 1o vicino nel {primo:.0%} delle righe')

## Generazione (k=4, prompt `new`)

Un TSV per embedding, con la ripresa dei notebook dei metodi. `sbert_mpnet_en` è già completo
(notebook 20).

In [ ]:
for emb in EMBEDDING:
    def righe_emb(emb=emb):
        for es in righe:
            esempi = [ESEMPI_TRAIN[v] for v, _ in VICINI[emb][es['row_id']]]
            documento = su.tronca_parole(su.prepara_documento(es['document']),
                                         su.MAX_PAROLE_DOCUMENTO_FEWSHOT)
            yield {**es, 'document': [{'role': 'user',
                                       'content': su.prompt_fewshot(esempi, documento, VARIANTE)}]}

    def genera(msg):
        return su.genera_ollama_nativo(msg, MODELLO, MAX_TOKENS, TEMPERATURE)[0]

    scrittore = su.ScrittoreRiassunti(percorso_riassunti(emb))
    su.ciclo_summarization(righe_emb(), scrittore, genera, etichetta=f'{emb} ', prepara=lambda m: m)
    scrittore.chiudi()

## Metriche

Stesse metriche del notebook 20. Per ogni embedding: la media, la differenza **appaiata**
rispetto a `sbert_mpnet_en` (la regola di decisione) e rispetto al controllo zero-shot k=0 del
notebook 20. In più la baseline *retrieval-only* (il riassunto del 1° vicino usato come
riassunto) e la contaminazione dagli esempi, con le stesse definizioni del notebook 20.

In [ ]:
import re

valutatore = su.crea_valutatore()
riferimenti = {r['row_id']: su.pulisci_riferimento(r['summary']) for r in righe}
predizioni = {emb: su.carica_riassunti(percorso_riassunti(emb)) for emb in EMBEDDING}
predizioni['k0'] = su.carica_riassunti(PILOTA_DIR / 'k0.tsv')
for emb in EMBEDDING:
    predizioni[f'retrieval_{emb}'] = {r['row_id']: ESEMPI_TRAIN[VICINI[emb][r['row_id']][0][0]]['summary']
                                      for r in righe}

coppie, righe_metriche = [], []
for nome, pred in predizioni.items():
    for rid, testo in pred.items():
        righe_metriche.append({'config': nome, 'row_id': rid,
                               **su.metriche_esempio(valutatore, testo, riferimenti[rid])})
        coppie.append((f'{nome}|{rid}', testo, riferimenti[rid]))
bs = su.calcola_bertscore_batch(coppie)
df = pd.DataFrame(righe_metriche)
df['bertscore_f1'] = [bs[f'{c}|{r}']['bertscore_f1'] for c, r in zip(df.config, df.row_id)]

METRICHE = ['rouge1_f1', 'rouge2_f1', 'rougeL_f1', 'meteor', 'bertscore_f1', 'parole_generate']
ordine = list(predizioni)
media = df.groupby('config')[METRICHE].mean().loc[ordine]
print('n per configurazione:', dict(df.groupby('config').size()))
print(media.round(4).to_string())

In [ ]:
def diff_appaiata(a, b, m):
    x = (df[df.config == a].set_index('row_id')[m] - df[df.config == b].set_index('row_id')[m]).dropna()
    return float(x.mean()), float(1.96 * x.std() / np.sqrt(len(x)))

confronti = {}
for emb in EMBEDDING:
    for base in ([RIFERIMENTO] if emb != RIFERIMENTO else []) + ['k0']:
        chiave = f'{emb}-{base}'
        confronti[chiave] = {m: diff_appaiata(emb, base, m) for m in METRICHE}
        print(f'{chiave:34s} ' + '  '.join(f'{m}: {mu:+.4f} ±{ic:.4f}' for m, (mu, ic)
                                           in confronti[chiave].items() if m != 'parole_generate'))

In [ ]:
def parole(t):
    return {w.lower() for w in re.findall(r'\b[A-Za-z]{3,}\b', t)}

def maiuscole(t):
    return set(re.findall(r'\b[A-Z][a-z]{2,}\b', t))

sorgente = {r['row_id']: parole(r['document']) for r in righe}
contaminazione, sim1 = {}, {}
for emb in EMBEDDING:
    n = []
    for r in righe:
        rid = r['row_id']
        ex = set().union(*(parole(ESEMPI_TRAIN[v]['document'] + ' ' + ESEMPI_TRAIN[v]['summary'])
                           for v, _ in VICINI[emb][rid]))
        n.append(sum(1 for w in maiuscole(predizioni[emb][rid]) if w.lower() not in sorgente[rid] and w.lower() in ex))
    contaminazione[emb] = float(np.mean(n))
    sim1[emb] = float(np.median([VICINI[emb][r['row_id']][0][1] for r in righe]))
    print(f'{emb:20s} contaminazione {contaminazione[emb]:.2f} parole/riassunto | '
          f'similarita\' del 1o vicino (mediana) {sim1[emb]:.3f} | ROUGE-1 F1 retrieval-only '
          f'{media.loc["retrieval_" + emb, "rouge1_f1"]:.3f}')

In [ ]:
# Esito della regola di decisione e riepilogo su disco
esito = {}
for emb in EMBEDDING:
    if emb == RIFERIMENTO:
        continue
    mu, ic = confronti[f'{emb}-{RIFERIMENTO}']['rouge1_f1']
    esito[emb] = ('migliore' if mu - ic > 0 else 'peggiore' if mu + ic < 0 else 'non distinguibile')
    print(f'{emb:20s} ROUGE-1 F1 vs {RIFERIMENTO}: {mu:+.4f} ±{ic:.4f} -> {esito[emb]}')

riepilogo = {'righe': 'val', 'n': N_PILOTA, 'seed': SEED, 'k': K, 'variante': VARIANTE,
             'modello': MODELLO, 'num_ctx': su.NUM_CTX_FEWSHOT,
             'media': media.round(5).to_dict(orient='index'),
             'confronti': {c: {m: {'media': round(mu, 5), 'ic95': round(ic, 5)} for m, (mu, ic) in v.items()}
                           for c, v in confronti.items()},
             'contaminazione_esempi': contaminazione, 'sim1_mediana': sim1,
             'esito_rispetto_a_' + RIFERIMENTO: esito}
EMB_DIR.mkdir(parents=True, exist_ok=True)
with open(RISULTATI_PATH, 'w', encoding='utf-8') as f:
    json.dump(riepilogo, f, indent=2, ensure_ascii=False)
print(RISULTATI_PATH)

## Conclusione

*(da compilare dopo l'esecuzione, applicando la regola di decisione fissata sopra)*